# Brain Tumour Segmentation — Attention U-Net
**Kaggle Notebook | T4 GPU | Run all cells top-to-bottom**

Pipeline: Install → Create modules → Find images → Grad-CAM pseudo-masks → Train Attention U-Net → Evaluate

## Step 1 — Install Packages

In [ ]:
import subprocess, sys

pkgs = [
    'albumentations==1.3.1',
    'timm==0.9.12',
    'opencv-python-headless',
    'reportlab',
    'fpdf2',
]
for pkg in pkgs:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True)
    print(('OK  ' if r.returncode == 0 else 'FAIL') + '  ' + pkg)

import albumentations, timm, cv2
print('albumentations:', albumentations.__version__)
print('timm          :', timm.__version__)
print('cv2           :', cv2.__version__)
print('Packages ready.')

## Step 2 — Create Segmentation Module Files

In [ ]:
import base64, os, sys

BASE = '/kaggle/working/BrainTumorAI'
os.makedirs(BASE + '/segmentation', exist_ok=True)
sys.path.insert(0, BASE)

# Each value is base64-encoded UTF-8 of the .py file content
_modules = {
    'segmentation/__init__.py': '',
    'segmentation/unet.py': 'IiIic2VnbWVudGF0aW9uL3VuZXQucHkg4oCUIFN0YW5kYXJkIFUtTmV0IChubyBhdHRlbnRpb24gZ2F0ZXMpLiIiIgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCgpjbGFzcyBDb252QmxvY2sobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbl9jaCwgb3V0X2NoKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmJsb2NrID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uQ29udjJkKGluX2NoLCBvdXRfY2gsIDMsIHBhZGRpbmc9MSwgYmlhcz1GYWxzZSksCiAgICAgICAgICAgIG5uLkJhdGNoTm9ybTJkKG91dF9jaCksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uQ29udjJkKG91dF9jaCwgb3V0X2NoLCAzLCBwYWRkaW5nPTEsIGJpYXM9RmFsc2UpLAogICAgICAgICAgICBubi5CYXRjaE5vcm0yZChvdXRfY2gpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOgogICAgICAgIHJldHVybiBzZWxmLmJsb2NrKHgpCgoKY2xhc3MgVU5ldChubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5uZWxzPTMsIG91dF9jaGFubmVscz0xLCBiYXNlX2ZpbHRlcnM9MTYsIGJpbGluZWFyPVRydWUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGYgPSBiYXNlX2ZpbHRlcnMKICAgICAgICBzZWxmLnBvb2wgPSBubi5NYXhQb29sMmQoMikKCiAgICAgICAgc2VsZi5lbmMxID0gQ29udkJsb2NrKGluX2NoYW5uZWxzLCBmKQogICAgICAgIHNlbGYuZW5jMiA9IENvbnZCbG9jayhmLCAgICAgZiAqIDIpCiAgICAgICAgc2VsZi5lbmMzID0gQ29udkJsb2NrKGYgKiAyLCBmICogNCkKICAgICAgICBzZWxmLmVuYzQgPSBDb252QmxvY2soZiAqIDQsIGYgKiA4KQogICAgICAgIHNlbGYuYm90dGxlbmVjayA9IENvbnZCbG9jayhmICogOCwgZiAqIDE2KQoKICAgICAgICBzZWxmLnVwNCAgPSBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9MiwgbW9kZT0nYmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUpCiAgICAgICAgc2VsZi5kZWM0ID0gQ29udkJsb2NrKGYgKiAxNiArIGYgKiA4LCBmICogOCkKICAgICAgICBzZWxmLnVwMyAgPSBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9MiwgbW9kZT0nYmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUpCiAgICAgICAgc2VsZi5kZWMzID0gQ29udkJsb2NrKGYgKiA4ICArIGYgKiA0LCBmICogNCkKICAgICAgICBzZWxmLnVwMiAgPSBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9MiwgbW9kZT0nYmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUpCiAgICAgICAgc2VsZi5kZWMyID0gQ29udkJsb2NrKGYgKiA0ICArIGYgKiAyLCBmICogMikKICAgICAgICBzZWxmLnVwMSAgPSBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9MiwgbW9kZT0nYmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUpCiAgICAgICAgc2VsZi5kZWMxID0gQ29udkJsb2NrKGYgKiAyICArIGYsICAgICAgZikKCiAgICAgICAgc2VsZi5vdXRjID0gbm4uQ29udjJkKGYsIG91dF9jaGFubmVscywgMSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBlMSA9IHNlbGYuZW5jMSh4KQogICAgICAgIGUyID0gc2VsZi5lbmMyKHNlbGYucG9vbChlMSkpCiAgICAgICAgZTMgPSBzZWxmLmVuYzMoc2VsZi5wb29sKGUyKSkKICAgICAgICBlNCA9IHNlbGYuZW5jNChzZWxmLnBvb2woZTMpKQogICAgICAgIGIgID0gc2VsZi5ib3R0bGVuZWNrKHNlbGYucG9vbChlNCkpCgogICAgICAgIGQ0ID0gc2VsZi5kZWM0KHRvcmNoLmNhdChbZTQsIHNlbGYudXA0KGIpXSwgIGRpbT0xKSkKICAgICAgICBkMyA9IHNlbGYuZGVjMyh0b3JjaC5jYXQoW2UzLCBzZWxmLnVwMyhkNCldLCBkaW09MSkpCiAgICAgICAgZDIgPSBzZWxmLmRlYzIodG9yY2guY2F0KFtlMiwgc2VsZi51cDIoZDMpXSwgZGltPTEpKQogICAgICAgIGQxID0gc2VsZi5kZWMxKHRvcmNoLmNhdChbZTEsIHNlbGYudXAxKGQyKV0sIGRpbT0xKSkKICAgICAgICByZXR1cm4gc2VsZi5vdXRjKGQxKQoKICAgIGRlZiBjb3VudF9wYXJhbWV0ZXJzKHNlbGYpOgogICAgICAgIHJldHVybiBzdW0ocC5udW1lbCgpIGZvciBwIGluIHNlbGYucGFyYW1ldGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZCkK',
    'segmentation/attention_unet.py': 'IiIic2VnbWVudGF0aW9uL2F0dGVudGlvbl91bmV0LnB5IOKAlCBBdHRlbnRpb24gVS1OZXQgKE9rdGF5IGV0IGFsLiAyMDE4KS4iIiIKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgoKY2xhc3MgQ29udkJsb2NrKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fY2gsIG91dF9jaCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5ibG9jayA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYyZChpbl9jaCwgb3V0X2NoLCAzLCBwYWRkaW5nPTEsIGJpYXM9RmFsc2UpLAogICAgICAgICAgICBubi5CYXRjaE5vcm0yZChvdXRfY2gpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkNvbnYyZChvdXRfY2gsIG91dF9jaCwgMywgcGFkZGluZz0xLCBiaWFzPUZhbHNlKSwKICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMmQob3V0X2NoKSwKICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICByZXR1cm4gc2VsZi5ibG9jayh4KQoKCmNsYXNzIEF0dGVudGlvbkdhdGUobm4uTW9kdWxlKToKICAgICIiIkFkZGl0aXZlIGF0dGVudGlvbiBnYXRlIOKAlCBnYXRlcyBza2lwIGNvbm5lY3Rpb24gdXNpbmcgZGVjb2RlciBzaWduYWwuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIEZfZywgRl9sLCBGX2ludCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5XX2cgPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MmQoRl9nLCAgIEZfaW50LCAxLCBiaWFzPVRydWUpLAogICAgICAgICAgICBubi5CYXRjaE5vcm0yZChGX2ludCksCiAgICAgICAgKQogICAgICAgIHNlbGYuV194ID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uQ29udjJkKEZfbCwgICBGX2ludCwgMSwgYmlhcz1UcnVlKSwKICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMmQoRl9pbnQpLAogICAgICAgICkKICAgICAgICBzZWxmLnBzaSA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYyZChGX2ludCwgMSwgMSwgYmlhcz1UcnVlKSwKICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMmQoMSksCiAgICAgICAgICAgIG5uLlNpZ21vaWQoKSwKICAgICAgICApCiAgICAgICAgc2VsZi5yZWx1ID0gbm4uUmVMVShpbnBsYWNlPVRydWUpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgZywgeCk6CiAgICAgICAgIyBnID0gZ2F0aW5nIHNpZ25hbCAoZGVjb2RlciksIHggPSBza2lwIGNvbm5lY3Rpb24gKGVuY29kZXIpCiAgICAgICAgZ191cCA9IEYuaW50ZXJwb2xhdGUoZywgc2l6ZT14LnNoYXBlWzI6XSwgbW9kZT0nYmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUpCiAgICAgICAgcHNpICA9IHNlbGYucmVsdShzZWxmLldfZyhnX3VwKSArIHNlbGYuV194KHgpKQogICAgICAgIHBzaSAgPSBzZWxmLnBzaShwc2kpICAgICAgICAgICMgKEIsMSxILFcpIGF0dGVudGlvbiBtYXAKICAgICAgICByZXR1cm4geCAqIHBzaSAgICAgICAgICAgICAgICAjIGdhdGVkIHNraXAKCgpjbGFzcyBBdHRlbnRpb25VTmV0KG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5fY2hhbm5lbHM9Mywgb3V0X2NoYW5uZWxzPTEsIGJhc2VfZmlsdGVycz0xNiwgYmlsaW5lYXI9VHJ1ZSk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgZiA9IGJhc2VfZmlsdGVycwogICAgICAgIHNlbGYucG9vbCA9IG5uLk1heFBvb2wyZCgyKQoKICAgICAgICAjIEVuY29kZXIKICAgICAgICBzZWxmLmVuYzEgICAgICA9IENvbnZCbG9jayhpbl9jaGFubmVscywgZikKICAgICAgICBzZWxmLmVuYzIgICAgICA9IENvbnZCbG9jayhmLCAgICAgZiAqIDIpCiAgICAgICAgc2VsZi5lbmMzICAgICAgPSBDb252QmxvY2soZiAqIDIsIGYgKiA0KQogICAgICAgIHNlbGYuZW5jNCAgICAgID0gQ29udkJsb2NrKGYgKiA0LCBmICogOCkKICAgICAgICBzZWxmLmJvdHRsZW5lY2sgPSBDb252QmxvY2soZiAqIDgsIGYgKiAxNikKCiAgICAgICAgIyBEZWNvZGVyIHdpdGggYXR0ZW50aW9uIGdhdGVzCiAgICAgICAgc2VsZi51cDQgID0gbm4uVXBzYW1wbGUoc2NhbGVfZmFjdG9yPTIsIG1vZGU9J2JpbGluZWFyJywgYWxpZ25fY29ybmVycz1UcnVlKQogICAgICAgIHNlbGYuYXR0NCA9IEF0dGVudGlvbkdhdGUoRl9nPWYgKiAxNiwgRl9sPWYgKiA4LCBGX2ludD1mICogNCkKICAgICAgICBzZWxmLmRlYzQgPSBDb252QmxvY2soZiAqIDE2ICsgZiAqIDgsIGYgKiA4KQoKICAgICAgICBzZWxmLnVwMyAgPSBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9MiwgbW9kZT0nYmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUpCiAgICAgICAgc2VsZi5hdHQzID0gQXR0ZW50aW9uR2F0ZShGX2c9ZiAqIDgsICBGX2w9ZiAqIDQsIEZfaW50PWYgKiAyKQogICAgICAgIHNlbGYuZGVjMyA9IENvbnZCbG9jayhmICogOCAgKyBmICogNCwgZiAqIDQpCgogICAgICAgIHNlbGYudXAyICA9IG5uLlVwc2FtcGxlKHNjYWxlX2ZhY3Rvcj0yLCBtb2RlPSdiaWxpbmVhcicsIGFsaWduX2Nvcm5lcnM9VHJ1ZSkKICAgICAgICBzZWxmLmF0dDIgPSBBdHRlbnRpb25HYXRlKEZfZz1mICogNCwgIEZfbD1mICogMiwgRl9pbnQ9ZikKICAgICAgICBzZWxmLmRlYzIgPSBDb252QmxvY2soZiAqIDQgICsgZiAqIDIsIGYgKiAyKQoKICAgICAgICBzZWxmLnVwMSAgPSBubi5VcHNhbXBsZShzY2FsZV9mYWN0b3I9MiwgbW9kZT0nYmlsaW5lYXInLCBhbGlnbl9jb3JuZXJzPVRydWUpCiAgICAgICAgc2VsZi5hdHQxID0gQXR0ZW50aW9uR2F0ZShGX2c9ZiAqIDIsICBGX2w9ZiwgICAgIEZfaW50PW1heChmIC8vIDIsIDEpKQogICAgICAgIHNlbGYuZGVjMSA9IENvbnZCbG9jayhmICogMiAgKyBmLCAgICAgIGYpCgogICAgICAgIHNlbGYub3V0YyA9IG5uLkNvbnYyZChmLCBvdXRfY2hhbm5lbHMsIDEpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgZTEgPSBzZWxmLmVuYzEoeCkKICAgICAgICBlMiA9IHNlbGYuZW5jMihzZWxmLnBvb2woZTEpKQogICAgICAgIGUzID0gc2VsZi5lbmMzKHNlbGYucG9vbChlMikpCiAgICAgICAgZTQgPSBzZWxmLmVuYzQoc2VsZi5wb29sKGUzKSkKICAgICAgICBiICA9IHNlbGYuYm90dGxlbmVjayhzZWxmLnBvb2woZTQpKQoKICAgICAgICBnNCA9IHNlbGYudXA0KGIpCiAgICAgICAgZDQgPSBzZWxmLmRlYzQodG9yY2guY2F0KFtzZWxmLmF0dDQoZzQsIGU0KSwgZzRdLCBkaW09MSkpCgogICAgICAgIGczID0gc2VsZi51cDMoZDQpCiAgICAgICAgZDMgPSBzZWxmLmRlYzModG9yY2guY2F0KFtzZWxmLmF0dDMoZzMsIGUzKSwgZzNdLCBkaW09MSkpCgogICAgICAgIGcyID0gc2VsZi51cDIoZDMpCiAgICAgICAgZDIgPSBzZWxmLmRlYzIodG9yY2guY2F0KFtzZWxmLmF0dDIoZzIsIGUyKSwgZzJdLCBkaW09MSkpCgogICAgICAgIGcxID0gc2VsZi51cDEoZDIpCiAgICAgICAgZDEgPSBzZWxmLmRlYzEodG9yY2guY2F0KFtzZWxmLmF0dDEoZzEsIGUxKSwgZzFdLCBkaW09MSkpCgogICAgICAgIHJldHVybiBzZWxmLm91dGMoZDEpCgogICAgZGVmIGNvdW50X3BhcmFtZXRlcnMoc2VsZik6CiAgICAgICAgcmV0dXJuIHN1bShwLm51bWVsKCkgZm9yIHAgaW4gc2VsZi5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkKQo=',
    'segmentation/losses.py': 'IiIic2VnbWVudGF0aW9uL2xvc3Nlcy5weSDigJQgTG9zcyBmdW5jdGlvbnMgZm9yIGJpbmFyeSB0dW1vdXIgc2VnbWVudGF0aW9uLiIiIgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCgpjbGFzcyBUdmVyc2t5TG9zcyhubi5Nb2R1bGUpOgogICAgIiIiVHZlcnNreSBsb3NzIOKAlCBwZW5hbGlzZXMgRlAgbW9yZSB0aGFuIEZOIHdoZW4gYWxwaGEgPiBiZXRhLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBhbHBoYT0wLjcsIGJldGE9MC4zLCBzbW9vdGg9MS4wKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmFscGhhICA9IGFscGhhCiAgICAgICAgc2VsZi5iZXRhICAgPSBiZXRhCiAgICAgICAgc2VsZi5zbW9vdGggPSBzbW9vdGgKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBsb2dpdHMsIHRhcmdldHMpOgogICAgICAgIHByb2JzICAgICA9IHRvcmNoLnNpZ21vaWQobG9naXRzKQogICAgICAgIHByb2JzX2YgICA9IHByb2JzLnZpZXcocHJvYnMuc2l6ZSgwKSwgICAgIC0xKQogICAgICAgIHRhcmdldHNfZiA9IHRhcmdldHMudmlldyh0YXJnZXRzLnNpemUoMCksIC0xKQogICAgICAgIFRQID0gKHByb2JzX2YgKiB0YXJnZXRzX2YpLnN1bSgxKQogICAgICAgIEZQID0gKHByb2JzX2YgKiAoMSAtIHRhcmdldHNfZikpLnN1bSgxKQogICAgICAgIEZOID0gKCgxIC0gcHJvYnNfZikgKiB0YXJnZXRzX2YpLnN1bSgxKQogICAgICAgIHR2ZXJza3kgPSAoVFAgKyBzZWxmLnNtb290aCkgLyAoCiAgICAgICAgICAgIFRQICsgc2VsZi5hbHBoYSAqIEZQICsgc2VsZi5iZXRhICogRk4gKyBzZWxmLnNtb290aCkKICAgICAgICByZXR1cm4gKDEgLSB0dmVyc2t5KS5tZWFuKCkKCgpjbGFzcyBEaWNlTG9zcyhubi5Nb2R1bGUpOgogICAgIiIiU29mdCBEaWNlIGxvc3MuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNtb290aD0xLjApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuc21vb3RoID0gc21vb3RoCgogICAgZGVmIGZvcndhcmQoc2VsZiwgbG9naXRzLCB0YXJnZXRzKToKICAgICAgICBwcm9icyAgICAgPSB0b3JjaC5zaWdtb2lkKGxvZ2l0cykKICAgICAgICBwcm9ic19mICAgPSBwcm9icy52aWV3KHByb2JzLnNpemUoMCksICAgICAtMSkKICAgICAgICB0YXJnZXRzX2YgPSB0YXJnZXRzLnZpZXcodGFyZ2V0cy5zaXplKDApLCAtMSkKICAgICAgICBpbnRlciA9IChwcm9ic19mICogdGFyZ2V0c19mKS5zdW0oMSkKICAgICAgICBkaWNlICA9ICgyICogaW50ZXIgKyBzZWxmLnNtb290aCkgLyAoCiAgICAgICAgICAgIHByb2JzX2Yuc3VtKDEpICsgdGFyZ2V0c19mLnN1bSgxKSArIHNlbGYuc21vb3RoKQogICAgICAgIHJldHVybiAoMSAtIGRpY2UpLm1lYW4oKQoKCmNsYXNzIEJvdW5kYXJ5TG9zcyhubi5Nb2R1bGUpOgogICAgIiIiQkNFIHdlaWdodGVkIDJ4IGF0IHR1bW91ciBib3VuZGFyeSBwaXhlbHMuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGJvdW5kYXJ5X3dlaWdodD0yLjAsIGtlcm5lbF9zaXplPTUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYm91bmRhcnlfd2VpZ2h0ID0gYm91bmRhcnlfd2VpZ2h0CiAgICAgICAgc2VsZi5rZXJuZWxfc2l6ZSAgICAgPSBrZXJuZWxfc2l6ZQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGxvZ2l0cywgdGFyZ2V0cyk6CiAgICAgICAgYmNlX21hcCA9IEYuYmluYXJ5X2Nyb3NzX2VudHJvcHlfd2l0aF9sb2dpdHMoCiAgICAgICAgICAgIGxvZ2l0cywgdGFyZ2V0cywgcmVkdWN0aW9uPSdub25lJykKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgcGFkICAgICAgPSBzZWxmLmtlcm5lbF9zaXplIC8vIDIKICAgICAgICAgICAgZGlsICAgICAgPSBGLm1heF9wb29sMmQodGFyZ2V0cywgIHNlbGYua2VybmVsX3NpemUsIHN0cmlkZT0xLCBwYWRkaW5nPXBhZCkKICAgICAgICAgICAgZXJvICAgICAgPSAtRi5tYXhfcG9vbDJkKC10YXJnZXRzLCBzZWxmLmtlcm5lbF9zaXplLCBzdHJpZGU9MSwgcGFkZGluZz1wYWQpCiAgICAgICAgICAgIGJvdW5kYXJ5ID0gKGRpbCAtIGVybykuY2xhbXAoMCwgMSkKICAgICAgICAgICAgd2VpZ2h0ICAgPSAxLjAgKyBzZWxmLmJvdW5kYXJ5X3dlaWdodCAqIGJvdW5kYXJ5CiAgICAgICAgcmV0dXJuIChiY2VfbWFwICogd2VpZ2h0KS5tZWFuKCkKCgpjbGFzcyBDb21iaW5lZFNlZ0xvc3Mobm4uTW9kdWxlKToKICAgICIiIgogICAgMC41ICogVHZlcnNreSArIDAuMyAqIERpY2UgKyAwLjIgKiBCb3VuZGFyeS4KICAgIGZvcndhcmQoKSByZXR1cm5zIGEgU0NBTEFSIHRlbnNvciAobm90IGEgdHVwbGUpLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgdHZlcnNreV9hbHBoYT0wLjcsIHR2ZXJza3lfYmV0YT0wLjMsCiAgICAgICAgYm91bmRhcnlfd2VpZ2h0PTIuMCwKICAgICAgICB3X3R2ZXJza3k9MC41LCB3X2RpY2U9MC4zLCB3X2JvdW5kYXJ5PTAuMiwKICAgICAgICBzbW9vdGg9MS4wLAogICAgKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnR2ZXJza3kgID0gVHZlcnNreUxvc3ModHZlcnNreV9hbHBoYSwgdHZlcnNreV9iZXRhLCBzbW9vdGgpCiAgICAgICAgc2VsZi5kaWNlICAgICA9IERpY2VMb3NzKHNtb290aCkKICAgICAgICBzZWxmLmJvdW5kYXJ5ID0gQm91bmRhcnlMb3NzKGJvdW5kYXJ5X3dlaWdodCkKICAgICAgICBzZWxmLndfdCA9IHdfdHZlcnNreQogICAgICAgIHNlbGYud19kID0gd19kaWNlCiAgICAgICAgc2VsZi53X2IgPSB3X2JvdW5kYXJ5CgogICAgZGVmIGZvcndhcmQoc2VsZiwgbG9naXRzLCB0YXJnZXRzKToKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBzZWxmLndfdCAqIHNlbGYudHZlcnNreShsb2dpdHMsIHRhcmdldHMpICsKICAgICAgICAgICAgc2VsZi53X2QgKiBzZWxmLmRpY2UobG9naXRzLCB0YXJnZXRzKSAgICArCiAgICAgICAgICAgIHNlbGYud19iICogc2VsZi5ib3VuZGFyeShsb2dpdHMsIHRhcmdldHMpCiAgICAgICAgKQo=',
    'segmentation/metrics.py': 'IiIic2VnbWVudGF0aW9uL21ldHJpY3MucHkg4oCUIFNlZ21lbnRhdGlvbiBldmFsdWF0aW9uIG1ldHJpY3MuIiIiCmltcG9ydCB0b3JjaAoKCmRlZiBjb21wdXRlX2FsbF9tZXRyaWNzKGxvZ2l0cywgdGFyZ2V0cywgdGhyZXNob2xkPTAuNDUpOgogICAgIiIiCiAgICBDb21wdXRlIERpY2UsIElvVSwgUHJlY2lzaW9uLCBSZWNhbGwgZnJvbSByYXcgbG9naXRzLgoKICAgIFBhcmFtZXRlcnMKICAgIC0tLS0tLS0tLS0KICAgIGxvZ2l0cyAgIDogKEIsMSxILFcpIHJhdyBtb2RlbCBvdXRwdXQg4oCUIHNpZ21vaWQgYXBwbGllZCBpbnRlcm5hbGx5CiAgICB0YXJnZXRzICA6IChCLDEsSCxXKSBiaW5hcnkgZmxvYXQgezAsMX0KICAgIHRocmVzaG9sZDogcHJvYmFiaWxpdHkgY3V0LW9mZgoKICAgIFJldHVybnMKICAgIC0tLS0tLS0KICAgIGRpY3Qg4oCUIGRpY2UsIGlvdSwgcHJlY2lzaW9uLCByZWNhbGwgKGZsb2F0cyBpbiBbMCwxXSkKICAgICIiIgogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgcHJvYnMgPSB0b3JjaC5zaWdtb2lkKGxvZ2l0cy5mbG9hdCgpKQogICAgICAgIHByZWQgID0gKHByb2JzID49IHRocmVzaG9sZCkuZmxvYXQoKS52aWV3KC0xKQogICAgICAgIHRhcmcgID0gdGFyZ2V0cy5mbG9hdCgpLnZpZXcoLTEpCgogICAgICAgIFRQID0gKHByZWQgKiB0YXJnKS5zdW0oKS5pdGVtKCkKICAgICAgICBGUCA9IChwcmVkICogKDEgLSB0YXJnKSkuc3VtKCkuaXRlbSgpCiAgICAgICAgRk4gPSAoKDEgLSBwcmVkKSAqIHRhcmcpLnN1bSgpLml0ZW0oKQoKICAgICAgICBzICAgICAgICAgPSAxZS02CiAgICAgICAgZGljZSAgICAgID0gKDIgKiBUUCArIHMpIC8gKDIgKiBUUCArIEZQICsgRk4gKyBzKQogICAgICAgIGlvdSAgICAgICA9IChUUCArIHMpICAgICAvIChUUCArIEZQICsgRk4gKyBzKQogICAgICAgIHByZWNpc2lvbiA9IChUUCArIHMpICAgICAvIChUUCArIEZQICsgcykKICAgICAgICByZWNhbGwgICAgPSAoVFAgKyBzKSAgICAgLyAoVFAgKyBGTiArIHMpCgogICAgcmV0dXJuIHsKICAgICAgICAnZGljZSc6ICAgICAgZmxvYXQoZGljZSksCiAgICAgICAgJ2lvdSc6ICAgICAgIGZsb2F0KGlvdSksCiAgICAgICAgJ3ByZWNpc2lvbic6IGZsb2F0KHByZWNpc2lvbiksCiAgICAgICAgJ3JlY2FsbCc6ICAgIGZsb2F0KHJlY2FsbCksCiAgICB9CgoKIyBBbGlhc2VzIGtlcHQgZm9yIGJhY2t3YXJkIGNvbXBhdGliaWxpdHkKZGVmIGRpY2Vfc2NvcmUocHJlZCwgdGFyZ2V0LCB0aHJlc2hvbGQ9MC40NSwgc21vb3RoPTEuMCk6CiAgICByZXR1cm4gY29tcHV0ZV9hbGxfbWV0cmljcyhwcmVkLCB0YXJnZXQsIHRocmVzaG9sZClbJ2RpY2UnXQoKCmRlZiBpb3Vfc2NvcmUocHJlZCwgdGFyZ2V0LCB0aHJlc2hvbGQ9MC40NSwgc21vb3RoPTEuMCk6CiAgICByZXR1cm4gY29tcHV0ZV9hbGxfbWV0cmljcyhwcmVkLCB0YXJnZXQsIHRocmVzaG9sZClbJ2lvdSddCg==',
    'segmentation/dataset.py': 'IiIic2VnbWVudGF0aW9uL2RhdGFzZXQucHkg4oCUIEJyYWluIHR1bW91ciBzZWdtZW50YXRpb24gZGF0YXNldC4iIiIKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhc2V0CmltcG9ydCBhbGJ1bWVudGF0aW9ucyBhcyBBCmZyb20gYWxidW1lbnRhdGlvbnMucHl0b3JjaCBpbXBvcnQgVG9UZW5zb3JWMgoKQ0xBU1NFUyAgPSBbJ2dsaW9tYScsICdtZW5pbmdpb21hJywgJ25vdHVtb3InLCAncGl0dWl0YXJ5J10KSU1HX0VYVFMgPSB7Jy5qcGcnLCAnLmpwZWcnLCAnLnBuZycsICcuYm1wJywgJy50aWYnLCAnLnRpZmYnfQpNRUFOICAgICA9IFswLjQ4NSwgMC40NTYsIDAuNDA2XQpTVEQgICAgICA9IFswLjIyOSwgMC4yMjQsIDAuMjI1XQoKCmNsYXNzIEJyYWluU2VnRGF0YXNldChEYXRhc2V0KToKICAgICIiIgogICAgTG9hZHMgTVJJIGltYWdlcyBhbmQgdGhlaXIgcHNldWRvLW1hc2sgcGFpcnMuCgogICAgUGFyYW1ldGVycwogICAgLS0tLS0tLS0tLQogICAgaW1hZ2VfZGlyICAgIDogZm9sZGVyIGNvbnRhaW5pbmcgY2xhc3Mgc3ViZm9sZGVycyAoZ2xpb21hLywgbWVuaW5naW9tYS8sIC4uLikKICAgIHBzZXVkb19jYWNoZSA6IGZvbGRlciB3aXRoIC5ucHkgb3IgLnBuZyBtYXNrcyAoc2FtZSBzdGVtIGFzIGltYWdlKQogICAgaW1hZ2Vfc2l6ZSAgIDogc3F1YXJlIHJlc2l6ZSB0YXJnZXQKICAgIHRyYWluICAgICAgICA6IFRydWUgLT4gYXVnbWVudGF0aW9uOyBGYWxzZSAtPiByZXNpemUgKyBub3JtYWxpc2Ugb25seQoKICAgIF9fZ2V0aXRlbV9fIHJldHVybnMKICAgIC0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIHsnaW1hZ2UnOiBGbG9hdFRlbnNvciAoMyxILFcpLCAnbWFzayc6IEZsb2F0VGVuc29yICgxLEgsVyl9CiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgaW1hZ2VfZGlyLCBwc2V1ZG9fY2FjaGUsIGltYWdlX3NpemU9MjI0LCB0cmFpbj1UcnVlKToKICAgICAgICBzZWxmLnBzZXVkb19jYWNoZSA9IFBhdGgocHNldWRvX2NhY2hlKQogICAgICAgIHNlbGYuaW1hZ2Vfc2l6ZSAgID0gaW1hZ2Vfc2l6ZQogICAgICAgIHNlbGYudHJhaW4gICAgICAgID0gdHJhaW4KCiAgICAgICAgaW1hZ2VfZGlyICAgID0gUGF0aChpbWFnZV9kaXIpCiAgICAgICAgc2VsZi5zYW1wbGVzID0gW10KCiAgICAgICAgIyBUcnkgY2xhc3Mgc3ViZm9sZGVycyBmaXJzdAogICAgICAgIGZvdW5kX2NsYXNzID0gRmFsc2UKICAgICAgICBmb3IgY2xzIGluIENMQVNTRVM6CiAgICAgICAgICAgIGNsc19kaXIgPSBpbWFnZV9kaXIgLyBjbHMKICAgICAgICAgICAgaWYgY2xzX2Rpci5leGlzdHMoKToKICAgICAgICAgICAgICAgIGZvdW5kX2NsYXNzID0gVHJ1ZQogICAgICAgICAgICAgICAgZm9yIHAgaW4gY2xzX2Rpci5yZ2xvYignKicpOgogICAgICAgICAgICAgICAgICAgIGlmIHAuaXNfZmlsZSgpIGFuZCBwLnN1ZmZpeC5sb3dlcigpIGluIElNR19FWFRTOgogICAgICAgICAgICAgICAgICAgICAgICBzZWxmLnNhbXBsZXMuYXBwZW5kKHApCgogICAgICAgICMgRmFsbGJhY2s6IGZsYXQgZGlyZWN0b3J5IHNjYW4KICAgICAgICBpZiBub3QgZm91bmRfY2xhc3M6CiAgICAgICAgICAgIGZvciBwIGluIGltYWdlX2Rpci5yZ2xvYignKicpOgogICAgICAgICAgICAgICAgaWYgcC5pc19maWxlKCkgYW5kIHAuc3VmZml4Lmxvd2VyKCkgaW4gSU1HX0VYVFM6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5zYW1wbGVzLmFwcGVuZChwKQoKICAgICAgICBpZiBub3Qgc2VsZi5zYW1wbGVzOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgICAgICdObyBpbWFnZXMgZm91bmQgaW46ICcgKyBzdHIoaW1hZ2VfZGlyKSArCiAgICAgICAgICAgICAgICAnICBFeHBlY3RlZCBzdWJmb2xkZXJzOiAnICsgJywgJy5qb2luKENMQVNTRVMpCiAgICAgICAgICAgICkKCiAgICAgICAgaWYgdHJhaW46CiAgICAgICAgICAgIHNlbGYudGZtID0gQS5Db21wb3NlKFsKICAgICAgICAgICAgICAgIEEuUmVzaXplKGltYWdlX3NpemUsIGltYWdlX3NpemUpLAogICAgICAgICAgICAgICAgQS5Ib3Jpem9udGFsRmxpcChwPTAuNSksCiAgICAgICAgICAgICAgICBBLlZlcnRpY2FsRmxpcChwPTAuMyksCiAgICAgICAgICAgICAgICBBLlJhbmRvbVJvdGF0ZTkwKHA9MC41KSwKICAgICAgICAgICAgICAgIEEuU2hpZnRTY2FsZVJvdGF0ZSgKICAgICAgICAgICAgICAgICAgICBzaGlmdF9saW1pdD0wLjEwLCBzY2FsZV9saW1pdD0wLjE1LAogICAgICAgICAgICAgICAgICAgIHJvdGF0ZV9saW1pdD0zMCwgcD0wLjUpLAogICAgICAgICAgICAgICAgQS5SYW5kb21CcmlnaHRuZXNzQ29udHJhc3QocD0wLjQpLAogICAgICAgICAgICAgICAgQS5HYXVzc05vaXNlKHA9MC4zKSwKICAgICAgICAgICAgICAgIEEuTm9ybWFsaXplKG1lYW49TUVBTiwgc3RkPVNURCksCiAgICAgICAgICAgICAgICBUb1RlbnNvclYyKCksCiAgICAgICAgICAgIF0pCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi50Zm0gPSBBLkNvbXBvc2UoWwogICAgICAgICAgICAgICAgQS5SZXNpemUoaW1hZ2Vfc2l6ZSwgaW1hZ2Vfc2l6ZSksCiAgICAgICAgICAgICAgICBBLk5vcm1hbGl6ZShtZWFuPU1FQU4sIHN0ZD1TVEQpLAogICAgICAgICAgICAgICAgVG9UZW5zb3JWMigpLAogICAgICAgICAgICBdKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBsZW4oc2VsZi5zYW1wbGVzKQoKICAgIGRlZiBfbG9hZF9tYXNrKHNlbGYsIHN0ZW0sIGgsIHcpOgogICAgICAgIG5weSA9IHNlbGYucHNldWRvX2NhY2hlIC8gKHN0ZW0gKyAnLm5weScpCiAgICAgICAgcG5nID0gc2VsZi5wc2V1ZG9fY2FjaGUgLyAoc3RlbSArICcucG5nJykKICAgICAgICBpZiBucHkuZXhpc3RzKCk6CiAgICAgICAgICAgIG1hc2sgPSBucC5sb2FkKHN0cihucHkpKS5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICBlbGlmIHBuZy5leGlzdHMoKToKICAgICAgICAgICAgbSAgICA9IGN2Mi5pbXJlYWQoc3RyKHBuZyksIGN2Mi5JTVJFQURfR1JBWVNDQUxFKQogICAgICAgICAgICBtYXNrID0gKG0gLyAyNTUuMCkuYXN0eXBlKG5wLmZsb2F0MzIpIGlmIG0gaXMgbm90IE5vbmUgZWxzZSBOb25lCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbWFzayA9IE5vbmUKICAgICAgICBpZiBtYXNrIGlzIE5vbmU6CiAgICAgICAgICAgIG1hc2sgPSBucC56ZXJvcygoaCwgdyksIGR0eXBlPW5wLmZsb2F0MzIpCiAgICAgICAgaWYgbWFzay5uZGltID09IDM6CiAgICAgICAgICAgIG1hc2sgPSBtYXNrWzosIDosIDBdCiAgICAgICAgcmV0dXJuIG1hc2sKCiAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaWR4KToKICAgICAgICBpbWdfcGF0aCA9IHNlbGYuc2FtcGxlc1tpZHhdCiAgICAgICAgYmdyICAgICAgPSBjdjIuaW1yZWFkKHN0cihpbWdfcGF0aCkpCiAgICAgICAgaWYgYmdyIGlzIE5vbmU6CiAgICAgICAgICAgIGJnciA9IG5wLnplcm9zKChzZWxmLmltYWdlX3NpemUsIHNlbGYuaW1hZ2Vfc2l6ZSwgMyksIGR0eXBlPW5wLnVpbnQ4KQogICAgICAgIGltZyAgICAgID0gY3YyLmN2dENvbG9yKGJnciwgY3YyLkNPTE9SX0JHUjJSR0IpCiAgICAgICAgaCwgdyAgICAgPSBpbWcuc2hhcGVbOjJdCiAgICAgICAgbWFzayAgICAgPSBzZWxmLl9sb2FkX21hc2soaW1nX3BhdGguc3RlbSwgaCwgdykKCiAgICAgICAgcmVzdWx0ICAgICAgID0gc2VsZi50Zm0oaW1hZ2U9aW1nLCBtYXNrPW1hc2spCiAgICAgICAgaW1hZ2VfdGVuc29yID0gcmVzdWx0WydpbWFnZSddLmZsb2F0KCkKICAgICAgICBtYXNrX3RlbnNvciAgPSByZXN1bHRbJ21hc2snXS5mbG9hdCgpCiAgICAgICAgaWYgbWFza190ZW5zb3IubmRpbSA9PSAyOgogICAgICAgICAgICBtYXNrX3RlbnNvciA9IG1hc2tfdGVuc29yLnVuc3F1ZWV6ZSgwKQogICAgICAgIHJldHVybiB7J2ltYWdlJzogaW1hZ2VfdGVuc29yLCAnbWFzayc6IG1hc2tfdGVuc29yfQo=',
    'segmentation/pseudo_masks.py': 'IiIic2VnbWVudGF0aW9uL3BzZXVkb19tYXNrcy5weSDigJQgR2VuZXJhdGUgcHNldWRvIG1hc2tzIGZyb20gR3JhZC1DQU0uIiIiCmltcG9ydCBjdjIKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKQ0xBU1NFUyAgPSBbJ2dsaW9tYScsICdtZW5pbmdpb21hJywgJ25vdHVtb3InLCAncGl0dWl0YXJ5J10KSU1HX0VYVFMgPSB7Jy5qcGcnLCAnLmpwZWcnLCAnLnBuZycsICcuYm1wJ30KCgpkZWYgYnVpbGRfcHNldWRvX2RhdGFzZXQoCiAgICBpbWFnZXNfZGlyLAogICAgb3V0cHV0X2RpciwKICAgIGdyYWRjYW1fZm4sCiAgICB0aHJlc2hvbGQgICAgPSAwLjQwLAogICAgc2tpcF9ub3R1bW9yID0gVHJ1ZSwKICAgIHNhdmVfbnB5ICAgICA9IFRydWUsCiAgICBzYXZlX3BuZyAgICAgPSBUcnVlLAogICAgdmVyYm9zZSAgICAgID0gVHJ1ZSwKKToKICAgICIiIgogICAgQnVpbGQgcHNldWRvLW1hc2sgZGF0YXNldCBmcm9tIEdyYWQtQ0FNIGFjdGl2YXRpb25zLgoKICAgIFBhcmFtZXRlcnMKICAgIC0tLS0tLS0tLS0KICAgIGltYWdlc19kaXIgICA6IHJvb3QgZm9sZGVyIGNvbnRhaW5pbmcgY2xhc3Mgc3ViZm9sZGVycwogICAgb3V0cHV0X2RpciAgIDogd2hlcmUgdG8gd3JpdGUgbWFzayBmaWxlcwogICAgZ3JhZGNhbV9mbiAgIDogY2FsbGFibGUoaW1nX3JnYl9od2MpIC0+IDItRCBmbG9hdCBhcnJheSBpbiBbMCwxXQogICAgdGhyZXNob2xkICAgIDogQ0FNIGJpbmFyaXNhdGlvbiB0aHJlc2hvbGQKICAgIHNraXBfbm90dW1vciA6IGlmIFRydWUsIHNhdmVzIHplcm8gbWFzayBmb3Igbm90dW1vciBjbGFzcwogICAgc2F2ZV9ucHkgICAgIDogc2F2ZSBmbG9hdDMyIC5ucHkgZmlsZXMKICAgIHNhdmVfcG5nICAgICA6IHNhdmUgdWludDggLnBuZyBmaWxlcyAoMCBvciAyNTUpCiAgICB2ZXJib3NlICAgICAgOiBwcmludCBwcm9ncmVzcwoKICAgIFJldHVybnMKICAgIC0tLS0tLS0KICAgIGNvdW50IG9mIG1hc2tzIHdyaXR0ZW4KICAgICIiIgogICAgaW1hZ2VzX2RpciA9IFBhdGgoaW1hZ2VzX2RpcikKICAgIG91dHB1dF9kaXIgPSBQYXRoKG91dHB1dF9kaXIpCiAgICBvdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBrZXJuZWxfY2xlYW4gPSBjdjIuZ2V0U3RydWN0dXJpbmdFbGVtZW50KGN2Mi5NT1JQSF9FTExJUFNFLCAoNywgNykpCiAgICBjb3VudCAgICAgICAgPSAwCgogICAgZm9yIGNscyBpbiBDTEFTU0VTOgogICAgICAgIGNsc19kaXIgICAgPSBpbWFnZXNfZGlyIC8gY2xzCiAgICAgICAgaXNfbm90dW1vciA9IChjbHMgPT0gJ25vdHVtb3InKQoKICAgICAgICBpZiBub3QgY2xzX2Rpci5leGlzdHMoKToKICAgICAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgICAgIHByaW50KCcgIFNraXBwaW5nICcgKyBjbHMgKyAnOiBmb2xkZXIgbm90IGZvdW5kJykKICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgaW1nX3BhdGhzID0gW3AgZm9yIHAgaW4gY2xzX2Rpci5yZ2xvYignKicpCiAgICAgICAgICAgICAgICAgICAgIGlmIHAuc3VmZml4Lmxvd2VyKCkgaW4gSU1HX0VYVFNdCiAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgcHJpbnQoJyAgJyArIGNscyArICc6ICcgKyBzdHIobGVuKGltZ19wYXRocykpICsgJyBpbWFnZXMnKQoKICAgICAgICBmb3IgaW1nX3BhdGggaW4gaW1nX3BhdGhzOgogICAgICAgICAgICBzdGVtID0gaW1nX3BhdGguc3RlbQoKICAgICAgICAgICAgIyBTa2lwIGlmIGFscmVhZHkgZ2VuZXJhdGVkCiAgICAgICAgICAgIGlmIHNhdmVfbnB5IGFuZCAob3V0cHV0X2RpciAvIChzdGVtICsgJy5ucHknKSkuZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBjb3VudCArPSAxCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBpZiBzYXZlX3BuZyBhbmQgbm90IHNhdmVfbnB5IGFuZCAob3V0cHV0X2RpciAvIChzdGVtICsgJy5wbmcnKSkuZXhpc3RzKCk6CiAgICAgICAgICAgICAgICBjb3VudCArPSAxCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgYmdyID0gY3YyLmltcmVhZChzdHIoaW1nX3BhdGgpKQogICAgICAgICAgICBpZiBiZ3IgaXMgTm9uZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIEgsIFcgPSBiZ3Iuc2hhcGVbOjJdCgogICAgICAgICAgICBpZiBpc19ub3R1bW9yIGFuZCBza2lwX25vdHVtb3I6CiAgICAgICAgICAgICAgICBtYXNrID0gbnAuemVyb3MoKEgsIFcpLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgaW1nX3JnYiA9IGN2Mi5jdnRDb2xvcihiZ3IsIGN2Mi5DT0xPUl9CR1IyUkdCKQogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIGNhbSA9IGdyYWRjYW1fZm4oaW1nX3JnYikKICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgICAgICAgICBpZiB2ZXJib3NlOgogICAgICAgICAgICAgICAgICAgICAgICBwcmludCgnICAgIEdyYWRDQU0gZmFpbGVkIGZvciAnICsgc3RlbSArICc6ICcgKyBzdHIoZSkpCiAgICAgICAgICAgICAgICAgICAgbWFzayA9IG5wLnplcm9zKChILCBXKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgY2FtID0gY3YyLnJlc2l6ZShjYW0uYXN0eXBlKG5wLmZsb2F0MzIpLCAoVywgSCkpCiAgICAgICAgICAgICAgICAgICAgbW4sIG14ID0gY2FtLm1pbigpLCBjYW0ubWF4KCkKICAgICAgICAgICAgICAgICAgICBpZiBteCA+IG1uOgogICAgICAgICAgICAgICAgICAgICAgICBjYW0gPSAoY2FtIC0gbW4pIC8gKG14IC0gbW4pCgogICAgICAgICAgICAgICAgICAgIGJpbmFyeSA9IChjYW0gPj0gdGhyZXNob2xkKS5hc3R5cGUobnAudWludDgpCiAgICAgICAgICAgICAgICAgICAgYmluYXJ5ID0gY3YyLm1vcnBob2xvZ3lFeChiaW5hcnksIGN2Mi5NT1JQSF9PUEVOLCAga2VybmVsX2NsZWFuKQogICAgICAgICAgICAgICAgICAgIGJpbmFyeSA9IGN2Mi5tb3JwaG9sb2d5RXgoYmluYXJ5LCBjdjIuTU9SUEhfQ0xPU0UsIGtlcm5lbF9jbGVhbikKCiAgICAgICAgICAgICAgICAgICAgIyBSZW1vdmUgYmxvYnMgc21hbGxlciB0aGFuIDIwMCBweAogICAgICAgICAgICAgICAgICAgIG5fYywgbGFiZWxzLCBzdGF0cywgXyA9IGN2Mi5jb25uZWN0ZWRDb21wb25lbnRzV2l0aFN0YXRzKGJpbmFyeSwgOCkKICAgICAgICAgICAgICAgICAgICBjbGVhbiA9IG5wLnplcm9zX2xpa2UoYmluYXJ5KQogICAgICAgICAgICAgICAgICAgIGZvciBsYmwgaW4gcmFuZ2UoMSwgbl9jKToKICAgICAgICAgICAgICAgICAgICAgICAgaWYgc3RhdHNbbGJsLCBjdjIuQ0NfU1RBVF9BUkVBXSA+PSAyMDA6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICBjbGVhbltsYWJlbHMgPT0gbGJsXSA9IDEKICAgICAgICAgICAgICAgICAgICBtYXNrID0gY2xlYW4uYXN0eXBlKG5wLmZsb2F0MzIpCgogICAgICAgICAgICBpZiBzYXZlX25weToKICAgICAgICAgICAgICAgIG5wLnNhdmUoc3RyKG91dHB1dF9kaXIgLyAoc3RlbSArICcubnB5JykpLCBtYXNrKQogICAgICAgICAgICBpZiBzYXZlX3BuZzoKICAgICAgICAgICAgICAgIGN2Mi5pbXdyaXRlKHN0cihvdXRwdXRfZGlyIC8gKHN0ZW0gKyAnLnBuZycpKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIChtYXNrICogMjU1KS5hc3R5cGUobnAudWludDgpKQogICAgICAgICAgICBjb3VudCArPSAxCgogICAgaWYgdmVyYm9zZToKICAgICAgICBwcmludCgnVG90YWwgbWFza3Mgd3JpdHRlbjogJyArIHN0cihjb3VudCkgKyAnIC0+ICcgKyBzdHIob3V0cHV0X2RpcikpCiAgICByZXR1cm4gY291bnQK',
}

for rel, b64str in _modules.items():
    fpath = os.path.join(BASE, rel)
    os.makedirs(os.path.dirname(fpath), exist_ok=True)
    content = base64.b64decode(b64str.encode()).decode('utf-8') if b64str else ''
    with open(fpath, 'w', encoding='utf-8') as f:
        f.write(content)
    print('OK:', rel)

print('All module files created.')

## Step 3 — Find Dataset Images

In [ ]:
import os
from pathlib import Path

CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']

print('Scanning /kaggle/input for image folders...')
IMAGES_DIR = None

for dirpath, dirs, files in os.walk('/kaggle/input'):
    imgs = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if len(imgs) > 10:
        print('  Found', len(imgs), 'images at:', dirpath)
    # Check if this dir has class subfolders
    found = [c for c in CLASSES if (Path(dirpath) / c).exists()]
    if len(found) >= 2:
        IMAGES_DIR = dirpath
        print('>> Using:', IMAGES_DIR)
        break
    # Check if this IS a class subfolder and parent has others
    parent = Path(dirpath).parent
    found_p = [c for c in CLASSES if (parent / c).exists()]
    if len(found_p) >= 2 and IMAGES_DIR is None:
        IMAGES_DIR = str(parent)
        print('>> Using parent:', IMAGES_DIR)
        break

if IMAGES_DIR is None:
    print('ERROR: Could not auto-detect dataset. Add brain-tumor-segmented or USE-Me Test dataset via Add Data.')
    IMAGES_DIR = '/kaggle/input/use-me-test/USE-Me Test'

# Verify and count
total = 0
for cls in CLASSES:
    p = Path(IMAGES_DIR) / cls
    n = len([x for x in p.rglob('*') if x.is_file()]) if p.exists() else 0
    total += n
    print('  {:15s}: {:4d} images'.format(cls, n))
print('  {:15s}: {:4d}'.format('TOTAL', total))
assert total > 0, 'No images found! Add a brain tumor dataset via the right panel -> Add Data.'
print('Dataset ready.')

## Step 4 — Load Classifier & Grad-CAM Engine
Used to generate pseudo-masks from unlabelled MRI images.

In [ ]:
import sys, os, logging, warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

sys.path.insert(0, '/kaggle/working/BrainTumorAI')
os.chdir('/kaggle/working/BrainTumorAI')

import timm

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 4
IMAGE_SIZE  = 224
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
ENS_WEIGHTS = {'efficientnet': 0.4, 'resnet_cbam': 0.3, 'densenet': 0.3}

SEARCH_PATHS = [
    '/kaggle/working/BrainTumorAI/checkpoints',
    '/kaggle/input/braintumorai-code/checkpoints',
    '/kaggle/input/brain-tumor-code/checkpoints',
]

# ── Model definitions (no app.py import) ─────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        m = max(c // r, 8)
        self.fc = nn.Sequential(
            nn.Linear(c, m, bias=False), nn.ReLU(True), nn.Linear(m, c, bias=False))
    def forward(self, x):
        B, C, H, W = x.shape
        a = torch.sigmoid(self.fc(x.mean([2,3])) + self.fc(x.amax([2,3])))
        return x * a.view(B, C, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        return x * torch.sigmoid(self.conv(
            torch.cat([x.mean(1, True), x.amax(1, True)], 1)))

class CBAM(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.ca = ChannelAttention(c, r)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

class ResNetCBAM(nn.Module):
    def __init__(self, nc, pretrained=False):
        super().__init__()
        b = timm.create_model('resnet50', pretrained=pretrained, num_classes=0, global_pool='')
        self.conv1=b.conv1; self.bn1=b.bn1; self.act1=b.act1; self.maxpool=b.maxpool
        self.layer1=b.layer1; self.layer2=b.layer2; self.layer3=b.layer3; self.layer4=b.layer4
        self.cbam3=CBAM(1024); self.cbam4=CBAM(2048)
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.head=nn.Sequential(
            nn.Dropout(0.4), nn.Linear(2048,512), nn.ReLU(True),
            nn.Dropout(0.2), nn.Linear(512, nc))
    def forward(self, x):
        x=self.act1(self.bn1(self.conv1(x))); x=self.maxpool(x)
        x=self.layer1(x); x=self.layer2(x)
        x=self.cbam3(self.layer3(x)); x=self.cbam4(self.layer4(x))
        return self.head(self.pool(x).flatten(1))

class MockEnsemble:
    def predict(self, tensor):
        a = tensor.cpu().numpy()
        seed = int((a.mean()*1e4 + a.std()*1e3) % 1e6)
        rng  = np.random.RandomState(seed % 100000)
        dom  = rng.randint(0, NUM_CLASSES)
        b    = rng.dirichlet(np.ones(NUM_CLASSES) * 0.3)
        b[dom] += rng.uniform(0.4, 0.65)
        probs = b / b.sum()
        return probs, {n: probs for n in ['efficientnet','resnet_cbam','densenet']}

def preprocess(image):
    if isinstance(image, np.ndarray):
        img = image.copy().astype(np.uint8)
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    elif isinstance(image, Image.Image):
        img = np.array(image.convert('RGB'), dtype=np.uint8)
    else:
        return None
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE)).astype(np.float32) / 255.0
    img = (img - _MEAN) / _STD
    return torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).float()

def _best_ckpt(folder):
    ckpts = list(Path(folder).glob('best_*.pt'))
    if not ckpts:
        return None
    def _auc(p):
        try: return float(p.stem.split('auc')[-1])
        except: return 0.
    return max(ckpts, key=_auc)

def load_model():
    fns = {
        'efficientnet': lambda nc: timm.create_model(
            'efficientnet_b3.ra2_in1k', pretrained=False, num_classes=nc),
        'resnet_cbam':  lambda nc: ResNetCBAM(nc),
        'densenet':     lambda nc: timm.create_model(
            'densenet121', pretrained=False, num_classes=nc),
    }
    for base_str in SEARCH_PATHS:
        base = Path(base_str)
        if not base.exists():
            continue
        loaded = {}
        for mname, mfn in fns.items():
            ckpt = _best_ckpt(base / mname)
            if ckpt is None:
                continue
            try:
                m   = mfn(NUM_CLASSES)
                raw = torch.load(ckpt, map_location=DEVICE, weights_only=False)['model_state_dict']
                try:
                    m.load_state_dict(raw, strict=True)
                except Exception:
                    fixed = {}
                    for k, v in raw.items():
                        nk = k[len('backbone.'):] if k.startswith('backbone.') else k
                        if v.dim() == 4 and 'cbam' in nk and 'ca.fc' in nk:
                            v = v.squeeze(-1).squeeze(-1)
                        fixed[nk] = v
                    m.load_state_dict(fixed, strict=False)
                m.eval()
                loaded[mname] = m.to(DEVICE)
            except Exception as e:
                print('  Could not load', mname, ':', e)
        if loaded:
            print('Loaded classifiers:', list(loaded.keys()))
            return loaded, 'live'
    print('No classifier checkpoints found — using mock (Grad-CAM will be random blobs)')
    return None, 'demo'

loaded_models, model_mode = load_model()
print('Model mode:', model_mode)

In [ ]:
# Define gradcam_fn used by pseudo-mask generator
if model_mode == 'live':
    try:
        from inference.gradcam import EnsembleGradCAM
        _clf = type('E', (), {})()
        _clf.models = loaded_models
        _cam_engine = EnsembleGradCAM(_clf, model_mode)
        _DUMMY = {
            'class':'glioma','class_label':'Glioma','confidence':0.85,
            'probabilities':{'glioma':0.85,'meningioma':0.05,'notumor':0.05,'pituitary':0.05},
            'individual':{}
        }
        def gradcam_fn(img_np):
            t = preprocess(img_np)
            if t is None:
                return np.zeros((14, 14), dtype=np.float32)
            result = _cam_engine.generate(t.to(DEVICE), _DUMMY, img_np)
            return result['cam']
    except Exception as e:
        print('GradCAM import failed:', e, '— using blob fallback')
        model_mode = 'demo'

if model_mode == 'demo':
    def gradcam_fn(img_np):
        H, W = img_np.shape[:2]
        cam  = np.zeros((H, W), dtype=np.float32)
        cy, cx = H // 2 + np.random.randint(-H//6, H//6), W // 2 + np.random.randint(-W//6, W//6)
        ry, rx = H // 5, W // 5
        Y, X   = np.ogrid[:H, :W]
        blob   = np.exp(-((Y-cy)**2/(2*ry**2) + (X-cx)**2/(2*rx**2)))
        cam    = (blob * 0.8 + np.random.rand(H, W) * 0.2).astype(np.float32)
        return cam

print('gradcam_fn ready — mode:', model_mode)

## Step 5 — Generate Pseudo Masks via Grad-CAM

In [ ]:
import os
sys.path.insert(0, '/kaggle/working/BrainTumorAI')
from segmentation.pseudo_masks import build_pseudo_dataset

PSEUDO_DIR = '/kaggle/working/BrainTumorAI/data/pseudo_masks'
os.makedirs(PSEUDO_DIR, exist_ok=True)

print('Generating pseudo masks for:', IMAGES_DIR)
print('Output:', PSEUDO_DIR)
n = build_pseudo_dataset(
    images_dir   = IMAGES_DIR,
    output_dir   = PSEUDO_DIR,
    gradcam_fn   = gradcam_fn,
    threshold    = 0.40,
    skip_notumor = True,
    save_npy     = True,
    save_png     = True,
    verbose      = True,
)
print('Done. Masks generated:', n)

## Step 6 — Preview Pseudo Masks

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = '/kaggle/working/NeuroScan_UNet'
os.makedirs(OUTPUT_DIR, exist_ok=True)

mask_files = sorted(Path(PSEUDO_DIR).glob('*.png'))[:8]
print('PNG masks found:', len(list(Path(PSEUDO_DIR).glob('*.png'))))
print('NPY masks found:', len(list(Path(PSEUDO_DIR).glob('*.npy'))))

if mask_files:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor='#0f172a')
    for ax, mf in zip(axes.flat, mask_files):
        m = cv2.imread(str(mf), cv2.IMREAD_GRAYSCALE)
        ax.imshow(m, cmap='hot')
        ax.set_title(mf.stem[:18], color='#00d4ff', fontsize=7)
        ax.axis('off')
    for ax in axes.flat[len(mask_files):]:
        ax.axis('off')
    plt.suptitle('Pseudo Masks (bright=tumour region)', color='white', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + '/pseudo_mask_preview.png', dpi=80,
                bbox_inches='tight', facecolor='#0f172a')
    plt.show()
    print('Preview saved.')
else:
    print('No PNG masks found — check Step 5 output.')

## Step 7 — CONFIG
Single source of truth for all hyperparameters.

In [ ]:
import os

CONFIG = {
    'images_dir'     : IMAGES_DIR,           # set in Step 3
    'pseudo_cache'   : '/kaggle/working/BrainTumorAI/data/pseudo_masks',
    'image_size'     : 224,
    'in_channels'    : 3,
    'val_split'      : 0.15,
    'architecture'   : 'attention_unet',
    'base_filters'   : 16,
    'bilinear'       : True,
    'epochs'         : 40,
    'batch_size'     : 8,
    'grad_accum'     : 2,
    'lr'             : 3e-4,
    'tversky_alpha'  : 0.7,
    'tversky_beta'   : 0.3,
    'boundary_weight': 2.0,
    'threshold'      : 0.45,
    'num_workers'    : 2,
    'seed'           : 42,
    'out_dir'        : '/kaggle/working/BrainTumorAI/checkpoints/unet',
    'vis_every'      : 10,
    'device'         : 'cuda' if torch.cuda.is_available() else 'cpu',
    'early_stop'     : 15,
}
os.makedirs(CONFIG['out_dir'], exist_ok=True)
for k, v in CONFIG.items():
    print('  {:20s}: {}'.format(k, v))

## Step 8 — Dataset & DataLoaders

In [ ]:
import time, warnings
import numpy as np
import torch
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, random_split

warnings.filterwarnings('ignore')

from segmentation.dataset import BrainSegDataset

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
DEVICE = torch.device(CONFIG['device'])

def make_ds(train):
    return BrainSegDataset(
        image_dir    = CONFIG['images_dir'],
        pseudo_cache = CONFIG['pseudo_cache'],
        image_size   = CONFIG['image_size'],
        train        = train,
    )

full_ds = make_ds(train=True)
n_total = len(full_ds)
n_val   = max(1, int(n_total * CONFIG['val_split']))
n_train = n_total - n_val

gen = torch.Generator().manual_seed(CONFIG['seed'])
tr_sub, va_sub = random_split(full_ds, [n_train, n_val], generator=gen)
va_sub.dataset = make_ds(train=False)

ldr_kw = dict(batch_size=CONFIG['batch_size'],
              num_workers=CONFIG['num_workers'],
              pin_memory=True)
train_loader = DataLoader(tr_sub, shuffle=True,  **ldr_kw)
val_loader   = DataLoader(va_sub, shuffle=False, **ldr_kw)

print('Train samples:', n_train)
print('Val   samples:', n_val)
print('Total samples:', n_total)

# Quick batch shape check
batch = next(iter(train_loader))
print('Image batch shape:', batch['image'].shape)
print('Mask  batch shape:', batch['mask'].shape)
print('DataLoaders ready.')

## Step 9 — Model, Loss & Optimizer

In [ ]:
from segmentation.attention_unet import AttentionUNet
from segmentation.unet           import UNet
from segmentation.losses         import CombinedSegLoss
from segmentation.metrics        import compute_all_metrics

arch = CONFIG['architecture'].lower()
if 'attention' in arch:
    model = AttentionUNet(
        in_channels  = CONFIG['in_channels'],
        base_filters = CONFIG['base_filters'],
        bilinear     = CONFIG['bilinear'],
    ).to(DEVICE)
else:
    model = UNet(
        in_channels  = CONFIG['in_channels'],
        base_filters = CONFIG['base_filters'],
        bilinear     = CONFIG['bilinear'],
    ).to(DEVICE)

criterion = CombinedSegLoss(
    tversky_alpha   = CONFIG['tversky_alpha'],
    tversky_beta    = CONFIG['tversky_beta'],
    boundary_weight = CONFIG['boundary_weight'],
    w_tversky  = 0.5,
    w_dice     = 0.3,
    w_boundary = 0.2,
)
optimizer = torch.optim.Adam(
    model.parameters(), lr=CONFIG['lr'], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7)
scaler = GradScaler()

print('Architecture :', arch)
print('Parameters   :', format(model.count_parameters(), ','))
print('Loss         : CombinedSegLoss (returns scalar)')
print('Scheduler    : CosineAnnealingWarmRestarts(T_0=10)')
print('AMP          : enabled')
print()
print('{:>4} | {:>8} {:>7} {:>6} | {:>8} {:>7} {:>6} | {:>9} | {:>5}'.format(
    'Ep','TrLoss','TrDice','TrIoU','VaLoss','VaDice','VaIoU','LR','s'))
print('-' * 85)

## Step 10 — Training Loop (40 epochs, AMP, gradient accumulation)

In [ ]:
import os, time
import matplotlib.pyplot as plt

def run_epoch(loader, train):
    model.train() if train else model.eval()
    tots  = {'loss': 0., 'dice': 0., 'iou': 0.}
    n     = 0
    accum = CONFIG['grad_accum']

    for step, batch in enumerate(loader):
        imgs  = batch['image'].to(DEVICE)
        masks = batch['mask'].to(DEVICE)

        if train:
            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, masks)   # scalar tensor
            scaler.scale(loss / accum).backward()
            if (step + 1) % accum == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        else:
            with torch.no_grad(), autocast():
                logits = model(imgs)
                loss   = criterion(logits, masks)

        bs  = imgs.size(0)
        met = compute_all_metrics(logits.detach().float(), masks, CONFIG['threshold'])
        tots['loss'] += loss.item() * bs
        tots['dice'] += met['dice'] * bs
        tots['iou']  += met['iou']  * bs
        n += bs

    return {k: v / max(n, 1) for k, v in tots.items()}


history = {'tr_loss':[],'tr_dice':[],'tr_iou':[],
           'val_loss':[],'val_dice':[],'val_iou':[]}
best_dice      = 0.0
best_ckpt      = None
no_improve     = 0
OUT_DIR        = CONFIG['out_dir']
BEST_PATH      = OUT_DIR + '/best_unet.pth'

optimizer.zero_grad()

for epoch in range(1, CONFIG['epochs'] + 1):
    t0  = time.time()
    tr  = run_epoch(train_loader, True)
    val = run_epoch(val_loader,   False)
    scheduler.step(epoch)
    lr_now = optimizer.param_groups[0]['lr']

    for k in ['loss', 'dice', 'iou']:
        history['tr_'  + k].append(tr[k])
        history['val_' + k].append(val[k])

    print('{:>4d} | {:>8.4f} {:>7.4f} {:>6.4f} | {:>8.4f} {:>7.4f} {:>6.4f} | {:>9.2e} | {:>5.1f}s'.format(
        epoch,
        tr['loss'], tr['dice'], tr['iou'],
        val['loss'], val['dice'], val['iou'],
        lr_now, time.time() - t0), flush=True)

    if val['dice'] > best_dice:
        best_dice  = val['dice']
        no_improve = 0
        torch.save({
            'epoch'            : epoch,
            'model_state_dict' : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice'         : best_dice,
            'val_iou'          : val['iou'],
            'config'           : {
                'architecture': CONFIG['architecture'],
                'in_channels' : CONFIG['in_channels'],
                'base_filters': CONFIG['base_filters'],
                'bilinear'    : CONFIG['bilinear'],
                'image_size'  : CONFIG['image_size'],
                'threshold'   : CONFIG['threshold'],
            },
        }, BEST_PATH)
        best_ckpt = BEST_PATH
        print('         *** New best val Dice: {:.4f} -> saved'.format(best_dice))
    else:
        no_improve += 1
        if no_improve >= CONFIG['early_stop']:
            print('Early stopping at epoch {} (no improvement for {} epochs)'.format(
                epoch, CONFIG['early_stop']))
            break

    # Visualise every N epochs
    if CONFIG['vis_every'] > 0 and (epoch % CONFIG['vis_every'] == 0 or epoch == 1):
        model.eval()
        fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor='#0f172a')
        mean_v = np.array([0.485, 0.456, 0.406])
        std_v  = np.array([0.229, 0.224, 0.225])
        with torch.no_grad():
            for vbatch in val_loader:
                imgs2 = vbatch['image'].to(DEVICE)
                probs = torch.sigmoid(model(imgs2).float()).cpu()
                for i in range(min(4, imgs2.size(0))):
                    t = imgs2[i].cpu().numpy()
                    axes[0][i].imshow((t.transpose(1,2,0)*std_v+mean_v).clip(0,1))
                    axes[0][i].set_title('ep{}'.format(epoch), color='#00d4ff', fontsize=8)
                    axes[0][i].axis('off')
                    axes[1][i].imshow(probs[i,0].numpy(), cmap='RdYlGn', vmin=0, vmax=1)
                    axes[1][i].set_title('dice={:.3f}'.format(val['dice']),
                                         color='#10b981', fontsize=8)
                    axes[1][i].axis('off')
                break
        for ax in axes.flat:
            ax.set_facecolor('#0f172a')
        plt.tight_layout()
        plt.savefig('{}/vis_ep{:03d}.png'.format(OUT_DIR, epoch),
                    dpi=80, bbox_inches='tight', facecolor='#0f172a')
        plt.show()
        plt.close()

print('=' * 85)
print('Training complete!  Best Val Dice: {:.4f}'.format(best_dice))
print('Checkpoint:', best_ckpt)
print('=' * 85)

## Step 11 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='#0f172a')
pairs = [('loss','#ef4444'), ('dice','#10b981'), ('iou','#3b82f6')]
for ax, (key, color) in zip(axes, pairs):
    ax.plot(history['tr_'  + key], color=color, lw=2, label='Train')
    ax.plot(history['val_' + key], color=color, lw=2, ls='--', alpha=0.7, label='Val')
    ax.set_title(key.upper(), color='white', fontsize=12)
    ax.set_facecolor('#0f172a')
    ax.tick_params(colors='#94a3b8')
    ax.legend(facecolor='#1e293b', labelcolor='white', fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor('#334155')

plt.suptitle('Attention U-Net Training  |  Best Val Dice: {:.4f}'.format(best_dice),
             color='white', fontsize=13)
plt.tight_layout()
curves_path = OUTPUT_DIR + '/training_curves.png'
plt.savefig(curves_path, dpi=120, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('Saved:', curves_path)

## Step 12 — Validation Visualisation
Original | Predicted mask | Overlay with green contour

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Load best checkpoint
ckpt  = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('Loaded best checkpoint  (epoch {}, val_dice={:.4f})'.format(
    ckpt['epoch'], ckpt['val_dice']))

mean_v = np.array([0.485, 0.456, 0.406])
std_v  = np.array([0.229, 0.224, 0.225])
THR    = CONFIG['threshold']

# Collect up to 8 val images
images_show, masks_show, preds_show = [], [], []
with torch.no_grad():
    for vbatch in val_loader:
        imgs2  = vbatch['image'].to(DEVICE)
        msks2  = vbatch['mask']
        probs  = torch.sigmoid(model(imgs2).float()).cpu()
        preds  = (probs >= THR).float()
        for i in range(imgs2.size(0)):
            images_show.append(imgs2[i].cpu().numpy())
            masks_show.append(msks2[i, 0].numpy())
            preds_show.append(preds[i, 0].numpy())
            if len(images_show) >= 8:
                break
        if len(images_show) >= 8:
            break

N    = len(images_show)
fig, axes = plt.subplots(3, N, figsize=(N * 3, 9), facecolor='#0f172a')
if N == 1:
    axes = axes.reshape(3, 1)

row_labels = ['Original', 'Pred Mask', 'Overlay']
for col in range(N):
    rgb  = (images_show[col].transpose(1,2,0) * std_v + mean_v).clip(0, 1)
    pred = preds_show[col]

    axes[0][col].imshow(rgb)
    axes[0][col].axis('off')

    axes[1][col].imshow(pred, cmap='hot', vmin=0, vmax=1)
    axes[1][col].axis('off')

    overlay = rgb.copy()
    overlay[pred == 1] = overlay[pred == 1] * 0.5 + np.array([0, 0.8, 0]) * 0.5
    cnts, _ = cv2.findContours(
        pred.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay_u8 = (overlay * 255).astype(np.uint8)
    cv2.drawContours(overlay_u8, cnts, -1, (0, 255, 0), 2)
    axes[2][col].imshow(overlay_u8)
    axes[2][col].axis('off')

for r, label in enumerate(row_labels):
    axes[r][0].set_ylabel(label, color='white', fontsize=11, rotation=90)

for ax in axes.flat:
    ax.set_facecolor('#0f172a')

plt.suptitle('Validation Results  |  Best Dice: {:.4f}'.format(best_dice),
             color='white', fontsize=14)
plt.tight_layout()
val_pred_path = OUTPUT_DIR + '/val_predictions.png'
plt.savefig(val_pred_path, dpi=100, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print('Saved:', val_pred_path)

## Step 13 — Final Metrics Summary

In [ ]:
import torch

model.eval()
all_dice, all_iou, all_prec, all_rec = [], [], [], []
with torch.no_grad():
    for vbatch in val_loader:
        imgs2 = vbatch['image'].to(DEVICE)
        msks2 = vbatch['mask'].to(DEVICE)
        with autocast():
            logits = model(imgs2)
        m = compute_all_metrics(logits.float(), msks2, CONFIG['threshold'])
        all_dice.append(m['dice'])
        all_iou.append(m['iou'])
        all_prec.append(m['precision'])
        all_rec.append(m['recall'])

print('=' * 50)
print('  FINAL VALIDATION METRICS')
print('=' * 50)
print('  Dice     : {:.4f}'.format(sum(all_dice)/len(all_dice)))
print('  IoU      : {:.4f}'.format(sum(all_iou)/len(all_iou)))
print('  Precision: {:.4f}'.format(sum(all_prec)/len(all_prec)))
print('  Recall   : {:.4f}'.format(sum(all_rec)/len(all_rec)))
print('=' * 50)
print('  Architecture :', CONFIG['architecture'])
print('  Parameters   :', format(model.count_parameters(), ','))
print('  Epochs trained:', len(history['val_dice']))
print('  Best Val Dice :', '{:.4f}'.format(best_dice))
print('=' * 50)

## Step 14 — Save Checkpoint to Output Tab
Download from Kaggle Output tab after this cell runs.

In [ ]:
import shutil, glob, os

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copy checkpoint
if best_ckpt and os.path.exists(best_ckpt):
    dest = OUTPUT_DIR + '/best_unet.pth'
    shutil.copy2(best_ckpt, dest)
    mb = os.path.getsize(dest) / 1_048_576
    print('Checkpoint saved: {} ({:.1f} MB)'.format(dest, mb))
else:
    print('WARNING: No checkpoint found.')

# Copy all vis images
for f in glob.glob(OUT_DIR + '/*.png'):
    shutil.copy2(f, OUTPUT_DIR)

# List output
print()
print('Files in Output tab:')
for f in sorted(glob.glob(OUTPUT_DIR + '/*')):
    print('  {:50s} {:5d} KB'.format(
        os.path.basename(f), os.path.getsize(f) // 1024))
print()
print('Download: Kaggle page -> Output tab -> click file -> Download')